#### 4.1 Loading the Libraries

The required libraries were imported for model development, evaluation, experiment tracking, and model persistence.

In [1]:
# Importing the required libraries

import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    VotingRegressor,
    StackingRegressor
)
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import mlflow
import mlflow.sklearn

#### 4.2 Loading the Prepared Data

The preprocessed training and testing datasets generated during data preparation were loaded for model development. Using the saved datasets ensures that the same train-test split and preprocessing transformations are maintained throughout the modeling workflow.

In [2]:
# Loading the prepared training and testing datasets

X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [4]:
# Checking the dimensions of the prepared datasets

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (800, 18)
X_test shape: (200, 18)
y_train shape: (800,)
y_test shape: (200,)


In [5]:
# Checking for missing values

print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())
print("Missing values in y_train:", y_train.isnull().sum())
print("Missing values in y_test:", y_test.isnull().sum())

Missing values in X_train: 0
Missing values in X_test: 0
Missing values in y_train: 0
Missing values in y_test: 0


#### 4.3 Establishing the Baseline Model

A baseline model was established to provide a reference point for evaluating the performance of the machine learning models.

The baseline predicts the mean historical ride cost from the training data for every observation in the test set. More advanced models should demonstrate meaningful improvement over this benchmark.

In [6]:
# Importing the baseline regression model

from sklearn.dummy import DummyRegressor

In [8]:
# Creating the baseline model

baseline_model = DummyRegressor(strategy='mean')

In [9]:
# Training the baseline model

baseline_model.fit(X_train, y_train)

,"strategy strategy: {""mean"", ""median"", ""quantile"", ""constant""}, default=""mean""Strategy to use to generate predictions.* ""mean"": always predicts the mean of the training set* ""median"": always predicts the median of the training set* ""quantile"": always predicts a specified quantile of the training set, provided with the quantile parameter.* ""constant"": always predicts a constant value that is provided by the user.",'mean'
,"constant constant: int or float or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
,"quantile quantile: float in [0.0, 1.0], default=NoneThe quantile to predict using the ""quantile"" strategy. A quantile of0.5 corresponds to the median, while 0.0 to the minimum and 1.0 to themaximum.",None
Name,Type,Value
"constant_ constant_: ndarray of shape (1, n_outputs)Mean or median or quantile of the training targets or constant valuegiven by the user.","ndarray[float64](1, 1)",[[370.73]]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](18,)","['num__Number_of_Riders','num__Number_of_Drivers', 'num__Number_of_Past_Rides',...,'cat__Time_of_Booking_Night', 'cat__Vehicle_Type_Economy','cat__Vehicle_Type_Premium']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,18
n_outputs_ n_outputs_: intNumber of outputs.,int,1


In [10]:
# Generating baseline predictions

baseline_predictions = baseline_model.predict(X_test)

In [11]:
# Calculating baseline performance

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))
baseline_r2 = r2_score(y_test, baseline_predictions)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline R²:", baseline_r2)

Baseline MAE: 160.96417751928144
Baseline RMSE: 191.15280997910116
Baseline R²: -0.0021555706797884433


## 4.4 Training the Individual Regression Models

The prepared training data was used to develop a range of regression models, covering linear, regularized, tree-based, boosting, and neural-network approaches.

MLflow was configured to track model parameters, performance metrics, and model artifacts throughout the development process. This provides a reproducible record of the experiments and supports systematic model comparison.

In [12]:
# Configuring MLflow experiment tracking

mlflow.set_tracking_uri("sqlite:///../mlflow.db")

mlflow.set_experiment("Ride Pricing Analytics")

2026/09/26 19:13:16 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/26 19:13:16 INFO mlflow.store.db.utils: Updating database tables
2026/09/26 19:13:20 INFO mlflow.tracking.fluent: Experiment with name 'Ride Pricing Analytics' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:///d:/Projects/ML and Data Science/Ride Pricing '
 'Analytics/notebooks/mlruns/1'), creation_time=1790430200062, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1790430200062, lifecycle_stage='active', name='Ride Pricing Analytics', tags={}, trace_location=None, workspace='default'>

In [13]:
# Checking the active MLflow experiment

experiment = mlflow.get_experiment_by_name("Ride Pricing Analytics")

print("Experiment Name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)

Experiment Name: Ride Pricing Analytics
Experiment ID: 1


### 4.4.2 Defining the Individual Models

A diverse set of regression models was selected to capture both linear and non-linear relationships within the ride-pricing data.

The initial model set includes linear regression, regularized regression, tree-based ensemble methods, gradient boosting, and a neural-network-based regression model. These models provide different learning approaches before hyperparameter tuning is applied to selected candidates.

In [14]:
# Defining the individual regression models

models = {
    'Linear Regression': LinearRegression(),

    'Ridge Regression': Ridge(alpha=1.0),

    'Lasso Regression': Lasso(
        alpha=0.01,
        max_iter=10000
    ),

    'Random Forest': RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    'Extra Trees': ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),

    'MLP Regressor': MLPRegressor(
        hidden_layer_sizes=(32, 16),
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42
    )
}

### 4.4.3 Training and Tracking Individual Models

Each individual regression model was trained using the prepared training dataset.

The model parameters and training performance were recorded in MLflow to maintain a reproducible experiment history. The held-out test dataset was not used for model selection at this stage and will be evaluated separately in the model evaluation stage.

In [16]:
# Training and tracking the individual models

trained_models = {}
training_results = []

for model_name, model in models.items():

    with mlflow.start_run(run_name=model_name):

        # Train the model
        model.fit(X_train, y_train)

        # Generate training predictions
        train_predictions = model.predict(X_train)

        # Calculate training metrics
        train_mae = mean_absolute_error(
            y_train,
            train_predictions
        )

        train_rmse = np.sqrt(
            mean_squared_error(
                y_train,
                train_predictions
            )
        )

        train_r2 = r2_score(
            y_train,
            train_predictions
        )

        # Store the trained model
        trained_models[model_name] = model

        # Store training results
        training_results.append({
            'Model': model_name,
            'Training MAE': train_mae,
            'Training RMSE': train_rmse,
            'Training R²': train_r2
        })

        # Log model parameters
        mlflow.log_params(model.get_params())

        # Log training metrics
        mlflow.log_metrics({
            'train_mae': train_mae,
            'train_rmse': train_rmse,
            'train_r2': train_r2
        })

training_results = pd.DataFrame(training_results)

training_results

,Model,Training MAE,Training RMSE,Training R²
0,Linear Regression,5.118590e+01,6.509354e+01,0.877577
1,Ridge Regression,5.117991e+01,6.509392e+01,0.877575
2,Lasso Regression,5.118327e+01,6.509357e+01,0.877577
3,Random Forest,1.942185e+01,2.557490e+01,0.981102
4,Extra Trees,8.868151e-13,1.136256e-12,1.000000
5,Gradient Boosting,4.082862e+01,5.193413e+01,0.922072
6,MLP Regressor,5.326579e+01,6.648277e+01,0.872295


In [17]:
# Creating the model directory

import os

os.makedirs("../models", exist_ok=True)

In [18]:
# Saving the trained models

for model_name, model in trained_models.items():

    file_name = (
        model_name
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    joblib.dump(
        model,
        f"../models/{file_name}.joblib"
    )

print("All trained models saved successfully.")

All trained models saved successfully.


### 4.4.4 Validating Saved Models

The trained models were saved as serialized artifacts so that they can be reused in subsequent evaluation, comparison, and explainability stages without retraining them.

In [19]:
# Checking the saved model artifacts

model_files = [
    file for file in os.listdir("../models")
    if file.endswith(".joblib")
]

print("Saved model artifacts:")

for file in model_files:
    print("-", file)

print("\nTotal saved models:", len(model_files))

Saved model artifacts:
- extra_trees.joblib
- gradient_boosting.joblib
- lasso_regression.joblib
- linear_regression.joblib
- mlp_regressor.joblib
- preprocessor.joblib
- random_forest.joblib
- ridge_regression.joblib

Total saved models: 8


#### 4.4.5 Adding Advanced Boosting Models

Two advanced gradient boosting models, XGBoost and LightGBM, were added to the model development process.

These models extend the initial set of regression approaches by providing efficient implementations of gradient boosting that can capture complex non-linear relationships between ride characteristics and historical ride costs.

The initial configurations were kept moderate to establish a computationally efficient benchmark before considering any hyperparameter tuning.

In [20]:
# Importing advanced boosting models

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [21]:
# Adding XGBoost and LightGBM to the model collection

models.update({
    'XGBoost': XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=2,
        objective='reg:squarederror'
    ),

    'LightGBM': LGBMRegressor(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=2,
        verbosity=-1
    )
})

### 4.4.6 Training and Tracking the Advanced Boosting Models

The XGBoost and LightGBM models were trained using the prepared training dataset.

Their model parameters and training performance were recorded in MLflow using the same experiment-tracking approach as the initial regression models. The test dataset remains reserved for the dedicated model evaluation stage.

In [22]:
# Selecting the advanced boosting models

advanced_models = {
    'XGBoost': models['XGBoost'],
    'LightGBM': models['LightGBM']
}

In [23]:
# Training and tracking the advanced boosting models

for model_name, model in advanced_models.items():

    with mlflow.start_run(run_name=model_name):

        # Train the model
        model.fit(X_train, y_train)

        # Generate training predictions
        train_predictions = model.predict(X_train)

        # Calculate training metrics
        train_mae = mean_absolute_error(
            y_train,
            train_predictions
        )

        train_rmse = np.sqrt(
            mean_squared_error(
                y_train,
                train_predictions
            )
        )

        train_r2 = r2_score(
            y_train,
            train_predictions
        )

        # Store the trained model
        trained_models[model_name] = model

        # Store training results
        training_results = pd.concat(
            [
                training_results,
                pd.DataFrame([{
                    'Model': model_name,
                    'Training MAE': train_mae,
                    'Training RMSE': train_rmse,
                    'Training R²': train_r2
                }])
            ],
            ignore_index=True
        )

        # Log model parameters
        mlflow.log_params(model.get_params())

        # Log training metrics
        mlflow.log_metrics({
            'train_mae': train_mae,
            'train_rmse': train_rmse,
            'train_r2': train_r2
        })

training_results

,Model,Training MAE,Training RMSE,Training R²
0,Linear Regression,5.118590e+01,6.509354e+01,0.877577
1,Ridge Regression,5.117991e+01,6.509392e+01,0.877575
2,Lasso Regression,5.118327e+01,6.509357e+01,0.877577
3,Random Forest,1.942185e+01,2.557490e+01,0.981102
4,Extra Trees,8.868151e-13,1.136256e-12,1.000000
5,Gradient Boosting,4.082862e+01,5.193413e+01,0.922072
6,MLP Regressor,5.326579e+01,6.648277e+01,0.872295
7,XGBoost,3.848183e+01,4.883583e+01,0.931093
8,LightGBM,1.884660e+01,2.494155e+01,0.982026


#### 4.4.7 Saving the Advanced Boosting Models

The trained XGBoost and LightGBM models were saved as model artifacts to ensure that the complete set of individual models can be reused in subsequent evaluation, comparison, and explainability stages without retraining.

In [24]:
# Saving the advanced boosting models

for model_name in ['XGBoost', 'LightGBM']:

    file_name = (
        model_name
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    joblib.dump(
        trained_models[model_name],
        f"../models/{file_name}.joblib"
    )

print("XGBoost and LightGBM models saved successfully.")

XGBoost and LightGBM models saved successfully.


In [25]:
# Verifying the complete set of saved model artifacts

model_files = [
    file for file in os.listdir("../models")
    if file.endswith(".joblib")
]

for file in sorted(model_files):
    print("-", file)

print("\nTotal joblib artifacts:", len(model_files))

- extra_trees.joblib
- gradient_boosting.joblib
- lasso_regression.joblib
- lightgbm.joblib
- linear_regression.joblib
- mlp_regressor.joblib
- preprocessor.joblib
- random_forest.joblib
- ridge_regression.joblib
- xgboost.joblib

Total joblib artifacts: 10


#### 4.4.8 Selecting Models for Hyperparameter Tuning

Hyperparameter tuning was restricted to selected non-linear models to balance model improvement with computational efficiency.

Tree-based and boosting models were prioritized because they are capable of capturing non-linear relationships in the ride-pricing data. Linear models and the neural-network benchmark were retained with their initial configurations rather than subjected to additional computationally intensive searches.

The selected models for tuning are Random Forest, Extra Trees, Gradient Boosting, XGBoost, and LightGBM.

#### 4.4.9 Defining Hyperparameter Search Spaces

A compact set of hyperparameter search spaces was defined for the selected non-linear models.

The search spaces focus on the parameters that have the greatest influence on model complexity and generalization while keeping the computational requirements manageable. A three-fold cross-validation strategy and a limited number of randomized combinations will be used during tuning.

In [26]:
# Importing the required tools for hyperparameter tuning

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

In [27]:
# Defining compact hyperparameter search spaces

param_distributions = {

    'Random Forest': {
        'n_estimators': randint(100, 251),
        'max_depth': [None, 10, 15, 20],
        'min_samples_split': randint(2, 11),
        'min_samples_leaf': randint(1, 5),
        'max_features': ['sqrt', 'log2']
    },

    'Extra Trees': {
        'n_estimators': randint(100, 251),
        'max_depth': [None, 10, 15, 20],
        'min_samples_split': randint(2, 11),
        'min_samples_leaf': randint(1, 5),
        'max_features': ['sqrt', 'log2']
    },

    'Gradient Boosting': {
        'n_estimators': randint(100, 251),
        'learning_rate': uniform(0.02, 0.08),
        'max_depth': randint(2, 5),
        'min_samples_split': randint(2, 11),
        'min_samples_leaf': randint(1, 5),
        'subsample': uniform(0.7, 0.3)
    },

    'XGBoost': {
        'n_estimators': randint(100, 251),
        'learning_rate': uniform(0.02, 0.08),
        'max_depth': randint(2, 6),
        'min_child_weight': randint(1, 6),
        'subsample': uniform(0.7, 0.3),
        'colsample_bytree': uniform(0.7, 0.3)
    },

    'LightGBM': {
        'n_estimators': randint(100, 251),
        'learning_rate': uniform(0.02, 0.08),
        'num_leaves': randint(15, 41),
        'max_depth': [-1, 5, 10, 15],
        'min_child_samples': randint(5, 21),
        'subsample': uniform(0.7, 0.3),
        'colsample_bytree': uniform(0.7, 0.3)
    }
}

#### 4.4.10 Hyperparameter Tuning

The selected non-linear models were tuned using `RandomizedSearchCV` with three-fold cross-validation.

A limited number of randomized parameter combinations was used to balance computational efficiency with sufficient exploration of the search spaces. Model selection during tuning was based on negative root mean squared error (`neg_root_mean_squared_error`), with the best cross-validation score used to identify the preferred configuration for each model.

In [28]:
# Defining the models selected for hyperparameter tuning

tuning_models = {
    'Random Forest': RandomForestRegressor(
        random_state=42,
        n_jobs=2
    ),

    'Extra Trees': ExtraTreesRegressor(
        random_state=42,
        n_jobs=2
    ),

    'Gradient Boosting': GradientBoostingRegressor(
        random_state=42
    ),

    'XGBoost': XGBRegressor(
        random_state=42,
        n_jobs=2,
        objective='reg:squarederror'
    ),

    'LightGBM': LGBMRegressor(
        random_state=42,
        n_jobs=2,
        verbosity=-1
    )
}

In [29]:
# Running hyperparameter tuning

tuned_models = {}
tuning_results = []

for model_name, model in tuning_models.items():

    print(f"\nTuning {model_name}...")

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions[model_name],
        n_iter=8,
        scoring='neg_root_mean_squared_error',
        cv=3,
        random_state=42,
        n_jobs=1,
        verbose=1
    )

    search.fit(X_train, y_train)

    tuned_models[model_name] = search.best_estimator_

    tuning_results.append({
        'Model': model_name,
        'Best CV RMSE': -search.best_score_,
        'Best Parameters': search.best_params_
    })

    print(f"Best CV RMSE: {-search.best_score_:.4f}")
    print("Best Parameters:")
    print(search.best_params_)


Tuning Random Forest...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best CV RMSE: 81.2930
Best Parameters:
{'max_depth': 20, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 157}

Tuning Extra Trees...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best CV RMSE: 88.9026
Best Parameters:
{'max_depth': 20, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 157}

Tuning Gradient Boosting...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best CV RMSE: 67.2516
Best Parameters:
{'learning_rate': np.float64(0.049963209507789), 'max_depth': 2, 'min_samples_leaf': 3, 'min_samples_split': 9, 'n_estimators': 120, 'subsample': np.float64(0.7468055921327309)}

Tuning XGBoost...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best CV RMSE: 69.0197
Best Parameters:
{'colsample_bytree': np.float64(0.935552788417904), 'learning_rate': np.float64(0.035973902572668776), 'max_de

In [30]:
# Creating a summary of the tuning results

tuning_results = pd.DataFrame(tuning_results)

tuning_results[
    ['Model', 'Best CV RMSE']
]

,Model,Best CV RMSE
0,Random Forest,81.293040
1,Extra Trees,88.902560
2,Gradient Boosting,67.251568
3,XGBoost,69.019743
4,LightGBM,69.373472


#### 4.4.11 Saving the Tuned Models

The best estimator identified for each tuned model was saved as a serialized model artifact. These tuned models will be reused during model evaluation and comparison without repeating the hyperparameter search.

In [31]:
# Saving the tuned models

for model_name, model in tuned_models.items():

    file_name = (
        model_name
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
        + "_tuned.joblib"
    )

    joblib.dump(
        model,
        f"../models/{file_name}"
    )

print("Tuned models saved successfully.")

Tuned models saved successfully.


In [32]:
# Checking the saved tuned model artifacts

tuned_model_files = [
    file for file in os.listdir("../models")
    if file.endswith("_tuned.joblib")
]

print("Saved tuned models:")

for file in tuned_model_files:
    print("-", file)

print("\nTotal tuned models:", len(tuned_model_files))

Saved tuned models:
- extra_trees_tuned.joblib
- gradient_boosting_tuned.joblib
- lightgbm_tuned.joblib
- random_forest_tuned.joblib
- xgboost_tuned.joblib

Total tuned models: 5


#### 4.4.12 Training the Ensemble Models

Ensemble regression models were developed to combine the predictive capabilities of multiple individual models.

Two ensemble approaches were considered: Voting Regression, which combines predictions from multiple models through averaging, and Stacking Regression, which uses a meta-model to learn how to combine predictions from the base estimators.

The ensemble models were trained using the prepared training dataset and will be evaluated on the held-out test dataset in the model evaluation stage.

In [33]:
# Defining the base estimators for ensemble models

ensemble_estimators = [
    ('ridge', Ridge(alpha=1.0)),

    ('gradient_boosting', tuned_models['Gradient Boosting']),

    ('xgboost', tuned_models['XGBoost']),

    ('lightgbm', tuned_models['LightGBM'])
]

Voting Regression

In [34]:
# Creating the Voting Regression model

voting_model = VotingRegressor(
    estimators=ensemble_estimators,
    n_jobs=1
)

Stacking Regression

In [35]:
# Creating the Stacking Regression model

stacking_model = StackingRegressor(
    estimators=ensemble_estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=3,
    n_jobs=1
)

In [36]:
# Training the ensemble models

ensemble_models = {
    'Voting Regression': voting_model,
    'Stacking Regression': stacking_model
}

for model_name, model in ensemble_models.items():

    print(f"Training {model_name}...")

    model.fit(X_train, y_train)

print("Ensemble models trained successfully.")

Training Voting Regression...
Training Stacking Regression...
Ensemble models trained successfully.


In [37]:
# Saving the ensemble models

for model_name, model in ensemble_models.items():

    file_name = (
        model_name
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
        + ".joblib"
    )

    joblib.dump(
        model,
        f"../models/{file_name}"
    )

print("Ensemble models saved successfully.")

Ensemble models saved successfully.


### 4.4.13 Validating the Complete Model Set

The complete collection of individual, tuned, and ensemble models was validated to ensure that the required model artifacts were successfully saved.

This final validation confirms that the trained models are available for reuse during model evaluation, comparison, and explainability without requiring retraining.

In [38]:
# Checking all saved model artifacts

model_files = [
    file for file in os.listdir("../models")
    if file.endswith(".joblib")
]

print("Saved model artifacts:")

for file in sorted(model_files):
    print("-", file)

print("\nTotal .joblib artifacts:", len(model_files))

Saved model artifacts:
- extra_trees.joblib
- extra_trees_tuned.joblib
- gradient_boosting.joblib
- gradient_boosting_tuned.joblib
- lasso_regression.joblib
- lightgbm.joblib
- lightgbm_tuned.joblib
- linear_regression.joblib
- mlp_regressor.joblib
- preprocessor.joblib
- random_forest.joblib
- random_forest_tuned.joblib
- ridge_regression.joblib
- stacking_regression.joblib
- voting_regression.joblib
- xgboost.joblib
- xgboost_tuned.joblib

Total .joblib artifacts: 17


In [39]:
# Creating a model-development summary

model_development_summary = pd.DataFrame({
    'Model Category': [
        'Baseline',
        'Individual Models',
        'Tuned Models',
        'Ensemble Models'
    ],
    'Count': [
        1,
        9,
        5,
        2
    ]
})

model_development_summary

,Model Category,Count
0,Baseline,1
1,Individual Models,9
2,Tuned Models,5
3,Ensemble Models,2
